In [0]:
%sql
-----  Creating new catalog, schema  ------ 
create catalog if not exists sql_youtube_practise;
use catalog sql_youtube_practise;
create schema if not exists sql;
use sql;
show current schema;

catalog,namespace
sql_youtube_practise,sql


#### Question1 - 
- ##### We have a table which stores data of multiple sections. 
- ##### Every section has 3 numbers we have to find top 4 numbers from any 2 sections(2 numbers each) whose addition should be maximum. 
- ##### So in this case we will choose section b where we have 19(10+9) then we need to choose either C or D because both has sum of 18 but in D we have 10 which is big from 9 so we will give priority to D.

In [0]:
%sql
DROP TABLE IF EXISTS section_data;
CREATE TABLE section_data (
    section VARCHAR(5),
    number INT
);
INSERT INTO section_data VALUES
('A', 5),
('A', 7),
('A', 10),
('B', 7),
('B', 9),
('B', 10),
('C', 9),
('C', 7),
('C', 9),
('D', 10),
('D', 3),
('D', 8);
SELECT * FROM section_data;

section,number
A,5
A,7
A,10
B,7
B,9
B,10
C,9
C,7
C,9
D,10


In [0]:
%sql
with cte as (
    select *, 
    row_number() over(partition by section order by number desc) as rnk
    from section_data
),
cte1 as (
    select *, 
        sum(number) over(partition by section) as sum,
        max(number) over(partition by section) as max_per_section
    from cte where rnk <=2
),
cte2 as (
    select *, 
        dense_rank() over(order by sum desc, max_per_section desc) as dnk
    from cte1
)
select * from cte2 
where dnk <=2 
order by dnk, section;

section,number,rnk,sum,max_per_section,dnk
B,10,1,19,10,1
B,9,2,19,10,1
D,10,1,18,10,2
D,8,2,18,10,2


#### Question2:
##### SQL Interview Questions - Booking Analysis --> Series of questions are attached below:

In [0]:
%sql
DROP TABLE IF EXISTS booking_table;
DROP TABLE IF EXISTS user_table;
CREATE TABLE booking_table (
    Booking_id       VARCHAR(3) NOT NULL,
    Booking_date     DATE NOT NULL,
    User_id          VARCHAR(3) NOT NULL,
    Line_of_business VARCHAR(6) NOT NULL
);
INSERT INTO booking_table (Booking_id, Booking_date, User_id, Line_of_business) VALUES
('b1','2022-03-23','u1','Flight'),
('b2','2022-03-27','u2','Flight'),
('b3','2022-03-28','u1','Hotel'),
('b4','2022-03-31','u4','Flight'),
('b5','2022-04-02','u1','Hotel'),
('b6','2022-04-02','u2','Flight'),
('b7','2022-04-06','u5','Flight'),
('b8','2022-04-06','u6','Hotel'),
('b9','2022-04-06','u2','Flight'),
('b10','2022-04-10','u1','Flight'),
('b11','2022-04-12','u4','Flight'),
('b12','2022-04-16','u1','Flight'),
('b13','2022-04-19','u2','Flight'),
('b14','2022-04-20','u5','Hotel'),
('b15','2022-04-22','u6','Flight'),
('b16','2022-04-26','u4','Hotel'),
('b17','2022-04-28','u2','Hotel'),
('b18','2022-04-30','u1','Hotel'),
('b19','2022-05-04','u4','Hotel'),
('b20','2022-05-06','u1','Flight');

CREATE TABLE user_table (
    User_id VARCHAR(3) NOT NULL,
    Segment VARCHAR(2) NOT NULL
);
INSERT INTO user_table (User_id, Segment) VALUES
('u1','s1'),
('u2','s1'),
('u3','s1'),
('u4','s2'),
('u5','s2'),
('u6','s3'),
('u7','s3'),
('u8','s3'),
('u9','s3'),
('u10','s3');
SELECT * FROM booking_table;

Booking_id,Booking_date,User_id,Line_of_business
b1,2022-03-23,u1,Flight
b2,2022-03-27,u2,Flight
b3,2022-03-28,u1,Hotel
b4,2022-03-31,u4,Flight
b5,2022-04-02,u1,Hotel
b6,2022-04-02,u2,Flight
b7,2022-04-06,u5,Flight
b8,2022-04-06,u6,Hotel
b9,2022-04-06,u2,Flight
b10,2022-04-10,u1,Flight


In [0]:
%sql
SELECT * FROM user_table;

User_id,Segment
u1,s1
u2,s1
u3,s1
u4,s2
u5,s2
u6,s3
u7,s3
u8,s3
u9,s3
u10,s3


##### Q1. Segment-Level Booking Analysis
Write an SQL query that gives the below output:
| Segment | Total_user_count | User_who_booked_flight_in_apr2022 |
|--------|------------------:|-----------------------------------:|
| s1     | 3                 | 2                                  |
| s2     | 2                 | 2                                  |
| s3     | 5                 | 1                                  |

In [0]:
%sql
------------------------------------ Approach 1 -------------------------------------------

with cte as (
    select * from booking_table 
    where Line_of_business = 'Flight' and Booking_date between '2022-04-01' and '2022-04-30'
),
cte1 as (
    select Segment, count(User_id) as count
    from user_table
    group by Segment
),
cte2 as (
    select count(distinct cte.User_id) as count_users, u.Segment from cte
    join user_table u
    on cte.User_id = u.User_id
    group by u.Segment
)
select * from cte1
join cte2
on cte1.Segment = cte2.Segment;

Segment,count,count_users,Segment
s1,3,2,s1
s2,2,2,s2
s3,5,1,s3


In [0]:
%sql
------------------------------------------- Approach 2 ---------------------------------------------

select u.Segment, count(distinct u.User_id) as no_of_users,
count(distinct
    case
        when b.Line_of_business = 'Flight' and b.Booking_date between '2022-04-01' and '2022-04-30' then u.User_id
    end
) as Users_who_booked_flight_on_Apr2022
from user_table u
left join booking_table b
on u.User_id = b.User_id
group by u.Segment;

Segment,no_of_users,Users_who_booked_flight_on_Apr2022
s2,2,2
s1,3,2
s3,5,1


##### Q2. First Booking Was a Hotel

Write a query to identify users whose **first booking was a hotel booking**.

In [0]:
%sql
with cte as (
select *,
row_number() over(partition by User_id order by Booking_date) as rnk
from booking_table
)
select * from cte
where rnk = 1 and Line_of_business = 'Hotel';

Booking_id,Booking_date,User_id,Line_of_business,rnk
b8,2022-04-06,u6,Hotel,1


##### Q3. Days Between First and Last Booking

Write a query to calculate the **number of days between the first and last booking of each user**.

In [0]:
%sql
select User_id, min(Booking_date), max(Booking_date), date_diff(day, min(Booking_date), max(Booking_date)) as diff
from booking_table
group by User_id

User_id,min(Booking_date),max(Booking_date),diff
u1,2022-03-23,2022-05-06,44
u2,2022-03-27,2022-04-28,32
u4,2022-03-31,2022-05-04,34
u5,2022-04-06,2022-04-20,14
u6,2022-04-06,2022-04-22,16


##### Q4. Flight and Hotel Bookings by Segment

Write a query to count the **number of flight and hotel bookings for each user segment for the year 2022**.

In [0]:
%sql
with cte as (
    select u.Segment, b.Line_of_business, count(Line_of_business) as Count from user_table as u
    join booking_table as b
    on u.User_id = b.User_id
    group by u.Segment, b.Line_of_business
)
select
    Segment,
    sum(case
        when Line_of_business = "Flight" then Count
    end) as Flight_Bookings,
    sum(case
        when Line_of_business = "Hotel" then Count
    end) as Hotel_Bookings
from cte
group by Segment;

Segment,Flight_Bookings,Hotel_Bookings
s1,8,4
s2,3,3
s3,1,1


##### Question3: Merge overlapping events in the same hall

In [0]:
%sql
DROP TABLE IF EXISTS hall_events;
CREATE TABLE hall_events (
    hall_id INT,
    start_date DATE,
    end_date DATE
);
INSERT INTO hall_events VALUES
(1, '2023-01-13', '2023-01-14'),
(1, '2023-01-14', '2023-01-17'),
(1, '2023-01-15', '2023-01-17'),
(1, '2023-01-18', '2023-01-25'),
(2, '2022-12-09', '2022-12-23'),
(2, '2022-12-13', '2022-12-17'),
(3, '2022-12-01', '2023-01-30');

SELECT * FROM hall_events ORDER BY hall_id, start_date;

hall_id,start_date,end_date
1,2023-01-13,2023-01-14
1,2023-01-14,2023-01-17
1,2023-01-15,2023-01-17
1,2023-01-18,2023-01-25
2,2022-12-09,2022-12-23
2,2022-12-13,2022-12-17
3,2022-12-01,2023-01-30


In [0]:
%sql
with cte as (
    select *,
        lag(end_date) over (partition by hall_id order by start_date) as prev_end_date
        from hall_events
),
cte1 as (
    select *,
        sum(
            case 
                when prev_end_date is null or start_date > prev_end_date 
                then 1 else 0 
            end
        ) over (partition by hall_id order by start_date) as grp
    from cte
)
select hall_id, min(start_date) as start_date, max(end_date) as end_date
from cte1
group by hall_id, grp

hall_id,start_date,end_date
1,2023-01-13,2023-01-17
1,2023-01-18,2023-01-25
2,2022-12-09,2022-12-23
3,2022-12-01,2023-01-30


#### Question4:
Write an SQL query to find the departments whose average salary is lower than the average salary of employees working in all the other departments combined.

In [0]:
%sql
DROP TABLE IF EXISTS emp;
CREATE TABLE emp (
    emp_id INT,
    emp_name VARCHAR(20),
    department_id INT,
    salary INT,
    manager_id INT,
    emp_age INT
);
INSERT INTO emp VALUES
(1, 'Ankit', 100, 10000, 4, 39),
(2, 'Mohit', 100, 15000, 5, 48),
(3, 'Vikas', 100, 10000, 4, 37),
(4, 'Rohit', 100, 5000, 2, 16),
(5, 'Mudit', 200, 12000, 6, 55),
(6, 'Agam', 200, 12000, 2, 14),
(7, 'Sanjay', 200, 9000, 2, 13),
(8, 'Ashish', 200, 5000, 2, 12),
(9, 'Mukesh', 300, 6000, 6, 51),
(10, 'Rakesh', 300, 7000, 6, 50);

SELECT * FROM emp;

emp_id,emp_name,department_id,salary,manager_id,emp_age
1,Ankit,100,10000,4,39
2,Mohit,100,15000,5,48
3,Vikas,100,10000,4,37
4,Rohit,100,5000,2,16
5,Mudit,200,12000,6,55
6,Agam,200,12000,2,14
7,Sanjay,200,9000,2,13
8,Ashish,200,5000,2,12
9,Mukesh,300,6000,6,51
10,Rakesh,300,7000,6,50


In [0]:
%sql
select department_id, avg(salary) as avg_dept_salary
from emp e1
group by department_id
having avg(salary) < (select avg(salary) from emp e2 where e1.department_id != e2.department_id)

department_id,avg_dept_salary
300,6500.0


#### Question5:
##### Calculate each employee's 
- total logins, 
- total logouts, 
- total check-in records, 
- latest login timestamps,
- latest logout timestamps,
- default phone number

In [0]:
%sql
DROP TABLE IF EXISTS employee_checkin_details;
DROP TABLE IF EXISTS employee_details;
CREATE TABLE employee_checkin_details (
    employeeid INT,
    entry_details STRING,
    timestamp_details TIMESTAMP
);
INSERT INTO employee_checkin_details VALUES
(1000, 'login',  '2023-06-16 01:00:15.34'),
(1000, 'login',  '2023-06-16 02:00:15.34'),
(1000, 'login',  '2023-06-16 03:00:15.34'),
(1000, 'logout', '2023-06-16 12:00:15.34'),
(1001, 'login',  '2023-06-16 01:00:15.34'),
(1001, 'login',  '2023-06-16 02:00:15.34'),
(1001, 'login',  '2023-06-16 03:00:15.34'),
(1001, 'logout', '2023-06-16 12:00:15.34');

CREATE TABLE employee_details (
    employeeid INT,
    phone_number INT,
    isdefault BOOLEAN
);
INSERT INTO employee_details VALUES
(1001, 9999, FALSE),
(1001, 1111, FALSE),
(1001, 2222, TRUE),
(1003, 3333, FALSE);

SELECT * FROM employee_checkin_details;

employeeid,entry_details,timestamp_details
1000,login,2023-06-16T01:00:15.340Z
1000,login,2023-06-16T02:00:15.340Z
1000,login,2023-06-16T03:00:15.340Z
1000,logout,2023-06-16T12:00:15.340Z
1001,login,2023-06-16T01:00:15.340Z
1001,login,2023-06-16T02:00:15.340Z
1001,login,2023-06-16T03:00:15.340Z
1001,logout,2023-06-16T12:00:15.340Z


In [0]:
%sql
SELECT * FROM employee_details;

employeeid,phone_number,isdefault
1001,9999,false
1001,1111,false
1001,2222,true
1003,3333,false


In [0]:
%sql
with cte as (
select e1.employeeid as emp1_id, e1.entry_details, e1.timestamp_details, e2.* from employee_checkin_details e1
left join employee_details e2
on e1.employeeid = e2.employeeid and e2.isdefault != false
),
cte1 as (
    select *,
        count(entry_details) over (partition by emp1_id) as total_entry,
        sum(
            case 
                when entry_details = 'login' then 1
            end 
        ) over (partition by emp1_id, entry_details) as total_logins,
        sum(
            case 
                when entry_details = 'logout' then 1
            end 
        ) over (partition by emp1_id, entry_details) as total_logouts,
        case 
            when entry_details = 'login' 
            then max(case when entry_details = 'login' then timestamp_details end)
            over (partition by emp1_id)
        end as latest_login,
        case 
            when entry_details = 'logout' 
            then max(case when entry_details = 'logout' then timestamp_details end)
            over (partition by emp1_id)
        end as latest_logout
    from cte
)
select
    emp1_id,
    max(phone_number) as employee_default_phone_number,
    max(total_entry),
    max(total_logins),
    max(total_logouts),
    max(latest_login),
    max(latest_logout)
from cte1
group by emp1_id;

emp1_id,employee_default_phone_number,max(total_entry),max(total_logins),max(total_logouts),max(latest_login),max(latest_logout)
1000,null,4,3,1,2023-06-16T03:00:15.340Z,2023-06-16T12:00:15.340Z
1001,2222,4,3,1,2023-06-16T03:00:15.340Z,2023-06-16T12:00:15.340Z


#### Question6:
- An organization wants to hire employees for junior and senior positions. 
- The organization has a total budget of $50,000. 
- It must first fill the senior positions and then fill the junior positions.

Write an SQL query to determine the maximum number of candidates that can be hired for each position without exceeding the budget.

In [0]:
%sql
DROP TABLE IF EXISTS candidates;
CREATE TABLE candidates (
    id INT PRIMARY KEY,
    positions VARCHAR(10) NOT NULL,
    salary INT NOT NULL
);

--------------------------------  Test Case 1  ---------------------------------------
-- INSERT INTO candidates VALUES
-- (1, 'junior', 5000),
-- (2, 'junior', 7000),
-- (3, 'junior', 7000),
-- (4, 'senior', 10000),
-- (5, 'senior', 30000),
-- (6, 'senior', 20000);


----------------------------------- Test case 2 ---------------------------------------------
-- insert into candidates values(20,'junior',10000);
-- insert into candidates values(30,'senior',15000);
-- insert into candidates values(40,'senior',30000);


----------------------------------- Test case 3 ---------------------------------------------
-- insert into candidates values(1,'junior',15000);
-- insert into candidates values(2,'junior',15000);
-- insert into candidates values(3,'junior',20000);
-- insert into candidates values(4,'senior',60000);


----------------------------------- Test case 4 ---------------------------------------------
insert into candidates values(10,'junior',10000);
insert into candidates values(40,'junior',10000);
insert into candidates values(20,'senior',15000);
insert into candidates values(30,'senior',30000);
insert into candidates values(50,'senior',15000);

SELECT * FROM candidates;

id,positions,salary
30,senior,30000
50,senior,15000
20,senior,15000
40,junior,10000
10,junior,10000


In [0]:
%sql
with cte as (
    select *,
        sum(salary) over(partition by positions order by salary rows between unbounded preceding and current row) as running_total
    from candidates
),
cte1 as (
    select count(*) as seniors, coalesce(sum(salary),0) as s_salary from cte
    where positions = "senior" and running_total < 50000
),
cte2 as (
    select count(*) as juniors from cte
    where positions = "junior" and running_total <= (50000 - (select s_salary from cte1))
)

select seniors, juniors from cte1, cte2;

seniors,juniors
2,2


#### Question7: Generate Available Job Positions and Identify Vacant Posts

Problem Statement - You are given two tables: `job_positions` and `job_employees`.

**Write a SQL query to generate one row for every available post and display the employee assigned to that post. If a post has no employee, display `'vacant'` instead.**

In [0]:
%sql
DROP TABLE IF EXISTS job_positions;
DROP TABLE IF EXISTS job_employees;
CREATE TABLE job_positions (
    id INT,
    title VARCHAR(100),
    groups VARCHAR(10),
    levels VARCHAR(10),
    payscale INT,
    totalpost INT
);
INSERT INTO job_positions VALUES
(1, 'General manager', 'A', 'l-15', 10000, 1),
(2, 'Manager', 'B', 'l-14', 9000, 5),
(3, 'Asst. Manager', 'C', 'l-13', 8000, 10);

CREATE TABLE job_employees (
    id INT,
    name VARCHAR(100),
    position_id INT
);
INSERT INTO job_employees VALUES
(1, 'John Smith', 1),
(2, 'Jane Doe', 2),
(3, 'Michael Brown', 2),
(4, 'Emily Johnson', 2),
(5, 'William Lee', 3),
(6, 'Jessica Clark', 3),
(7, 'Christopher Harris', 3),
(8, 'Olivia Wilson', 3),
(9, 'Daniel Martinez', 3),
(10, 'Sophia Miller', 3);

SELECT * FROM job_positions;

id,title,groups,levels,payscale,totalpost
1,General manager,A,l-15,10000,1
2,Manager,B,l-14,9000,5
3,Asst. Manager,C,l-13,8000,10


In [0]:
%sql
SELECT * FROM job_employees;

id,name,position_id
1,John Smith,1
2,Jane Doe,2
3,Michael Brown,2
4,Emily Johnson,2
5,William Lee,3
6,Jessica Clark,3
7,Christopher Harris,3
8,Olivia Wilson,3
9,Daniel Martinez,3
10,Sophia Miller,3


In [0]:
%sql
with recursive cte as (
    select id, title, groups, levels, payscale, totalpost, 1 as rn from job_positions
    union all
    select id, title, groups, levels, payscale, totalpost, rn + 1 from cte
    where rn + 1 <= totalpost
),
cte1 as (
    select *,
        row_number() over (partition by position_id order by id) as rank
    from job_employees
)
select cte.id, cte.title, cte.groups, cte.levels, cte.payscale, coalesce(cte1.name, 'vacant') as name from cte
left join cte1
on cte.id = cte1.position_id and cte.rn = cte1.rank
order by cte.id;


id,title,groups,levels,payscale,name
1,General manager,A,l-15,10000,John Smith
2,Manager,B,l-14,9000,Jane Doe
2,Manager,B,l-14,9000,Michael Brown
2,Manager,B,l-14,9000,Emily Johnson
2,Manager,B,l-14,9000,vacant
2,Manager,B,l-14,9000,vacant
3,Asst. Manager,C,l-13,8000,William Lee
3,Asst. Manager,C,l-13,8000,Jessica Clark
3,Asst. Manager,C,l-13,8000,Christopher Harris
3,Asst. Manager,C,l-13,8000,Olivia Wilson


#### Question8 - 
##### Write a SQL query to find cities where not a single order was returned

In [0]:
%sql
drop table if exists namaste_orders;
drop table if exists namaste_returns;

create table namaste_orders
(
    order_id int,
    city varchar(10),
    sales int
);

create table namaste_returns
(
    order_id int,
    return_reason varchar(20)
);

insert into namaste_orders values
(1, 'Mysore' , 100), (2, 'Mysore' , 200), (3, 'Bangalore' , 250), (4, 'Bangalore' , 150), (5, 'Mumbai' , 300), (6, 'Mumbai' , 500), (7, 'Mumbai' , 800);

insert into namaste_returns values
(3,'wrong item'),(6,'bad quality'),(7,'wrong item');

select * from namaste_orders;

order_id,city,sales
1,Mysore,100
2,Mysore,200
3,Bangalore,250
4,Bangalore,150
5,Mumbai,300
6,Mumbai,500
7,Mumbai,800


In [0]:
%sql
select * from namaste_returns;

order_id,return_reason
3,wrong item
6,bad quality
7,wrong item


In [0]:
%sql
------------------------------   Using join + group  -------------------------------
-- select o.city, count(r.order_id) as count from namaste_orders as o
-- left join namaste_returns as r
-- on o.order_id = r.order_id
-- group by o.city
-- having count = 0;

------------------------------  Using Sub-Query  -----------------------------------
select distinct city from namaste_orders where city not in (
    select distinct city from namaste_orders where order_id in (
        select order_id from namaste_returns
    )
);

city
Mysore


#### Question9 - 
##### Write a SQL to find start & end location of each customer

In [0]:
%sql
drop table if exists travel_data;
CREATE TABLE travel_data (
    customer VARCHAR(10),
    start_loc VARCHAR(50),
    end_loc VARCHAR(50)
);

INSERT INTO travel_data (customer, start_loc, end_loc) VALUES
    ('c1', 'New York', 'Lima'),
    ('c1', 'London', 'New York'),
    ('c1', 'Lima', 'Sao Paulo'),
    ('c1', 'Sao Paulo', 'New Delhi'),
    ('c2', 'Mumbai', 'Hyderabad'),
    ('c2', 'Surat', 'Pune'),
    ('c2', 'Hyderabad', 'Surat'),
    ('c3', 'Kochi', 'Kurnool'),
    ('c3', 'Lucknow', 'Agra'),
    ('c3', 'Agra', 'Jaipur'),
    ('c3', 'Jaipur', 'Kochi');

select * from travel_data;

customer,start_loc,end_loc
c1,New York,Lima
c1,London,New York
c1,Lima,Sao Paulo
c1,Sao Paulo,New Delhi
c2,Mumbai,Hyderabad
c2,Surat,Pune
c2,Hyderabad,Surat
c3,Kochi,Kurnool
c3,Lucknow,Agra
c3,Agra,Jaipur


In [0]:
%sql
select 
    customer,
    max(
        case 
            when start_loc not in (
                select end_loc from travel_data t2
                where t2.customer = t1.customer
            ) then start_loc 
        end 
    ) as starting_location,
    max(
        case
            when end_loc not in (
                select start_loc from travel_data t2
                where t2.customer = t1.customer
            ) then end_loc
        end 
    ) as ending_location
from travel_data t1
group by customer
order by customer;

customer,starting_location,ending_location
c1,London,New Delhi
c2,Mumbai,Pune
c3,Lucknow,Kurnool


#### Question9: SKU Price History and Next Month Calculation
You are given a table named `sku` that stores the price history of each SKU. 
Find price of each id at the start of next month.

**Table: `sku`**

| sku_id | price_date | price |
|--------|------------|-------|
| 1 | 2023-01-01 | 10 |
| 1 | 2023-02-15 | 15 |
| 1 | 2023-03-03 | 18 |
| 1 | 2023-03-27 | 15 |
| 1 | 2023-04-06 | 20 |

### Expected Output

| sku_id | price | next_month |
|--------|-------|------------|
| 1 | 10 | 2023-01-01 |
| 1 | 15 | 2023-03-01 |
| 1 | 15 | 2023-04-01 |
| 1 | 20 | 2023-05-01 |

In [0]:
%sql
drop table if exists sku;
create table sku 
(
    sku_id int,
    price_date date ,
    price int
);
insert into sku values (1,'2023-01-01',10),(1,'2023-02-15',15),(1,'2023-03-03',18),(1,'2023-03-27',15),(1,'2023-04-06',20);

select * from sku;

sku_id,price_date,price
1,2023-01-01,10
1,2023-02-15,15
1,2023-03-03,18
1,2023-03-27,15
1,2023-04-06,20


In [0]:
%sql
with cte as (
    select * from sku where date_part('day', price_date) = 01
), cte1 as (
    select * from sku 
    where price_date in (
        select max(price_date) from sku group by date_part('month', price_date)
    )
)
select sku_id,
    price,
    to_date(date_trunc('month', dateadd(month, 1, price_date))) as next_month
from cte1
union all
select sku_id, price, price_date from cte
order by next_month;

sku_id,price,next_month
1,10,2023-01-01
1,10,2023-02-01
1,15,2023-03-01
1,15,2023-04-01
1,20,2023-05-01



##### Question10: Split Full Names into First, Middle, and Last Names

Write a SQL query to split each customer's full name into three columns: `first_name`, `second_name`, and `last_name`.

Follow these rules:

1. If the name contains no spaces, treat the entire name as `first_name`. Set `second_name` and `last_name` to `NULL`.
2. If the name contains one space, split it into `first_name` and `last_name`. Set `second_name` to `NULL`.
3. If the name contains two or more spaces, extract the first word as `first_name`, the second word as `second_name`, and the remaining final part as `last_name`.

In [0]:
%sql
drop table if exists customers;
create table customers  (customer_name varchar(30));
insert into customers values ('Ankit Bansal'),('Vishal Pratap Singh'),('Michael'); 
select * from customers;

customer_name
Ankit Bansal
Vishal Pratap Singh
Michael


In [0]:
%sql
with cte as (
    select *,
        len(customer_name) - len(replace(customer_name, ' ', '')) as no_of_spaces,
        charindex(' ', customer_name) as first_space_position,
        charindex(' ', customer_name, (charindex(' ', customer_name) + 1)) as second_space_position
    from customers
)
select *,
    case 
        when no_of_spaces = 0 then customer_name
        else substring(customer_name, 1, first_space_position - 1)
    end as first_name,
    case
        when no_of_spaces <= 1 then null
        else substring(customer_name, first_space_position + 1, second_space_position - first_space_position - 1)
    end as second_name,
    case
        when no_of_spaces = 0 then null
        when no_of_spaces = 1 then substring(customer_name, first_space_position + 1, len(customer_name)-first_space_position)
        when no_of_spaces > 1 then substring(customer_name, second_space_position + 1, len(customer_name)-second_space_position)
    end as last_name
from cte

customer_name,no_of_spaces,first_space_position,second_space_position,first_name,second_name,last_name
Ankit Bansal,1,6,0,Ankit,null,Bansal
Vishal Pratap Singh,2,7,14,Vishal,Pratap,Singh
Michael,0,0,0,Michael,null,null


##### Question 11 -
Please consider the following as the categories for getting the distribution:
- No Social Interaction (No messages, emojis or gifts sent during the game)
- One sided interaction (Messages, emojis or gifts sent during the game by only one player)
- Both sided Interaction without custom_typed_messages
- Both sided interaction with custom_typed_messages from at least one player

In [0]:
%sql
drop table if exists user_interactions;
CREATE TABLE user_interactions (
    user_id varchar(10),
    event varchar(15),
    event_date DATE,
    interaction_type varchar(15),
    game_id varchar(10),
    event_time TIME
);

-- Insert the data
INSERT INTO user_interactions 
VALUES
('abc', 'game_start', '2024-01-01', null, 'ab0000', '10:00:00'),
('def', 'game_start', '2024-01-01', null, 'ab0000', '10:00:00'),
('def', 'send_emoji', '2024-01-01', 'emoji1', 'ab0000', '10:03:20'),
('def', 'send_message', '2024-01-01', 'preloaded_quick', 'ab0000', '10:03:49'),
('abc', 'send_gift', '2024-01-01', 'gift1', 'ab0000', '10:04:40'),
('abc', 'game_end', '2024-01-01', NULL, 'ab0000', '10:10:00'),
('def', 'game_end', '2024-01-01', NULL, 'ab0000', '10:10:00'),
('abc', 'game_start', '2024-01-01', null, 'ab9999', '10:00:00'),
('def', 'game_start', '2024-01-01', null, 'ab9999', '10:00:00'),
('abc', 'send_message', '2024-01-01', 'custom_typed', 'ab9999', '10:02:43'),
('abc', 'send_gift', '2024-01-01', 'gift1', 'ab9999', '10:04:40'),
('abc', 'game_end', '2024-01-01', NULL, 'ab9999', '10:10:00'),
('def', 'game_end', '2024-01-01', NULL, 'ab9999', '10:10:00'),
('abc', 'game_start', '2024-01-01', null, 'ab1111', '10:00:00'),
('def', 'game_start', '2024-01-01', null, 'ab1111', '10:00:00'),
('abc', 'game_end', '2024-01-01', NULL, 'ab1111', '10:10:00'),
('def', 'game_end', '2024-01-01', NULL, 'ab1111', '10:10:00'),
('abc', 'game_start', '2024-01-01', null, 'ab1234', '10:00:00'),
('def', 'game_start', '2024-01-01', null, 'ab1234', '10:00:00'),
('abc', 'send_message', '2024-01-01', 'custom_typed', 'ab1234', '10:02:43'),
('def', 'send_emoji', '2024-01-01', 'emoji1', 'ab1234', '10:03:20'),
('def', 'send_message', '2024-01-01', 'preloaded_quick', 'ab1234', '10:03:49'),
('abc', 'send_gift', '2024-01-01', 'gift1', 'ab1234', '10:04:40'),
('abc', 'game_end', '2024-01-01', NULL, 'ab1234', '10:10:00'),
('def', 'game_end', '2024-01-01', NULL, 'ab1234', '10:10:00');


num_affected_rows,num_inserted_rows
25,25


In [0]:
%sql
select * from user_interactions;

---- Following this below table, we can assume the categories to be falling as -
---- ab0000 - Both sided Interaction without custom_typed_messages
---- ab9999 - One sided interaction (Messages, emojis or gifts sent during the game by only one player)
---- ab1111 - No Social Interaction (No messages, emojis or gifts sent during the game)
---- ab1234 - Both sided interaction with custom_typed_messages from at least one player

user_id,event,event_date,interaction_type,game_id,event_time
abc,game_start,2024-01-01,null,ab0000,10:00:00
def,game_start,2024-01-01,null,ab0000,10:00:00
def,send_emoji,2024-01-01,emoji1,ab0000,10:03:20
def,send_message,2024-01-01,preloaded_quick,ab0000,10:03:49
abc,send_gift,2024-01-01,gift1,ab0000,10:04:40
abc,game_end,2024-01-01,null,ab0000,10:10:00
def,game_end,2024-01-01,null,ab0000,10:10:00
abc,game_start,2024-01-01,null,ab9999,10:00:00
def,game_start,2024-01-01,null,ab9999,10:00:00
abc,send_message,2024-01-01,custom_typed,ab9999,10:02:43


In [0]:
%sql
select game_id,
    case 
        when count(interaction_type) = 0 then 'No social interaction'
        when count(distinct case when interaction_type is not null then user_id end) = 1 then 'One sided interaction'
        when count(distinct case when interaction_type is not null then user_id end) > 1 and
            count(distinct case when interaction_type = 'custom_typed' then user_id end) = 0 then 'Both sided interaction without custom_typed_messages'
        when count(distinct case when interaction_type is not null then user_id end) > 1 and
            count(distinct case when interaction_type = 'custom_typed' then user_id end) >= 1 then 'Both sided interaction with custom_typed_messages'
    end as game_type
from user_interactions
group by game_id;

game_id,game_type
ab0000,Both sided interaction without custom_typed_messages
ab9999,One sided interaction
ab1111,No social interaction
ab1234,Both sided interaction with custom_typed_messages


#### Question12:
Write an SQL query to find the supplier ID, product ID & starting date of record_date for which stock quantity is less than 50 for two or more consecutive days.

In [0]:
%sql
-- Create the table
drop table if exists stock;
CREATE TABLE stock (
    supplier_id INT,
    product_id INT,
    stock_quantity INT,
    record_date DATE
);

-- Insert the data
delete from stock;
INSERT INTO stock (supplier_id, product_id, stock_quantity, record_date)
VALUES
    (1, 1, 60, '2022-01-01'),
    (1, 1, 40, '2022-01-02'),
    (1, 1, 35, '2022-01-03'),
    (1, 1, 45, '2022-01-04'),
 (1, 1, 51, '2022-01-06'),
 (1, 1, 55, '2022-01-09'),
 (1, 1, 25, '2022-01-10'),
    (1, 1, 48, '2022-01-11'),
 (1, 1, 45, '2022-01-15'),
    (1, 1, 38, '2022-01-16'),
    (1, 2, 45, '2022-01-08'),
    (1, 2, 40, '2022-01-09'),
    (2, 1, 45, '2022-01-06'),
    (2, 1, 55, '2022-01-07'),
    (2, 2, 45, '2022-01-08'),
 (2, 2, 48, '2022-01-09'),
    (2, 2, 35, '2022-01-10'),
 (2, 2, 52, '2022-01-15'),
    (2, 2, 23, '2022-01-16');

select * from stock;

supplier_id,product_id,stock_quantity,record_date
1,1,60,2022-01-01
1,1,40,2022-01-02
1,1,35,2022-01-03
1,1,45,2022-01-04
1,1,51,2022-01-06
1,1,55,2022-01-09
1,1,25,2022-01-10
1,1,48,2022-01-11
1,1,45,2022-01-15
1,1,38,2022-01-16


In [0]:
%sql
with cte as (
    select *,
        date_sub(record_date, (row_number() over (partition by supplier_id, product_id order by record_date))) as rn
    from stock
    where stock_quantity < 50
)
select
    supplier_id,
    product_id,
    min(record_date) as starting_date
from cte
group by supplier_id, product_id, rn
having count(*) >= 2
order by supplier_id, product_id;

supplier_id,product_id,starting_date
1,1,2022-01-15
1,1,2022-01-10
1,1,2022-01-02
1,2,2022-01-08
2,2,2022-01-08


#### Question 13:
Write an SQL query to find unique city pairs and their distances, eliminating reverse-direction duplicates. If the distance between two cities differs in the two directions, or if the reverse-direction record does not exist, retain the record for investigation.

In [0]:
%sql
drop table if exists city_distance;
CREATE TABLE city_distance
(
    distance INT,
    source VARCHAR(512),
    destination VARCHAR(512)
);
INSERT INTO city_distance(distance, source, destination) VALUES ('100', 'New Delhi', 'Panipat');
INSERT INTO city_distance(distance, source, destination) VALUES ('200', 'Ambala', 'New Delhi');
INSERT INTO city_distance(distance, source, destination) VALUES ('150', 'Bangalore', 'Mysore');
INSERT INTO city_distance(distance, source, destination) VALUES ('150', 'Mysore', 'Bangalore');
INSERT INTO city_distance(distance, source, destination) VALUES ('250', 'Mumbai', 'Pune');
INSERT INTO city_distance(distance, source, destination) VALUES ('250', 'Pune', 'Mumbai');
INSERT INTO city_distance(distance, source, destination) VALUES ('2500', 'Chennai', 'Bhopal');
INSERT INTO city_distance(distance, source, destination) VALUES ('2500', 'Bhopal', 'Chennai');
INSERT INTO city_distance(distance, source, destination) VALUES ('60', 'Tirupati', 'Tirumala');
INSERT INTO city_distance(distance, source, destination) VALUES ('80', 'Tirumala', 'Tirupati');

select * from city_distance;

distance,source,destination
100,New Delhi,Panipat
80,Tirumala,Tirupati
60,Tirupati,Tirumala
150,Mysore,Bangalore
150,Bangalore,Mysore
200,Ambala,New Delhi
2500,Bhopal,Chennai
2500,Chennai,Bhopal
250,Pune,Mumbai
250,Mumbai,Pune


In [0]:
%sql
select c1.*, c2.*
from city_distance as c1
left join city_distance as c2
on c1.source = c2.destination and c1.destination = c2.source
where c2.distance is null or c1.distance != c2.distance or c1.source < c2.source;

distance,source,destination,distance,source,destination
100,New Delhi,Panipat,null,null,null
80,Tirumala,Tirupati,60,Tirupati,Tirumala
60,Tirupati,Tirumala,80,Tirumala,Tirupati
150,Bangalore,Mysore,150,Mysore,Bangalore
200,Ambala,New Delhi,null,null,null
2500,Bhopal,Chennai,2500,Chennai,Bhopal
250,Mumbai,Pune,250,Pune,Mumbai


#### Question 14:
- Calculate the cumulative total of cases from all previous months (excluding the current month).
- Calculate the percentage contribution of the current month's cases compared to the cumulative total of previous months.

In [0]:
%sql
-- Create table
DROP TABLE IF EXISTS covid_cases;

CREATE TABLE covid_cases (
    record_date DATE,
    cases_count INT
);

-- Insert all records
INSERT INTO covid_cases (record_date, cases_count) VALUES
('2021-01-01',66),('2021-01-02',41),('2021-01-03',54),('2021-01-04',68),('2021-01-05',16),('2021-01-06',90),('2021-01-07',34),('2021-01-08',84),('2021-01-09',71),('2021-01-10',14),('2021-01-11',48),('2021-01-12',72),('2021-01-13',55),
('2021-02-01',38),('2021-02-02',57),('2021-02-03',42),('2021-02-04',61),('2021-02-05',25),('2021-02-06',78),('2021-02-07',33),('2021-02-08',93),('2021-02-09',62),('2021-02-10',15),('2021-02-11',52),('2021-02-12',76),('2021-02-13',45),
('2021-03-01',27),('2021-03-02',47),('2021-03-03',36),('2021-03-04',64),('2021-03-05',29),('2021-03-06',81),('2021-03-07',32),('2021-03-08',89),('2021-03-09',63),('2021-03-10',19),('2021-03-11',53),('2021-03-12',78),('2021-03-13',49),
('2021-04-01',39),('2021-04-02',58),('2021-04-03',44),('2021-04-04',65),('2021-04-05',30),('2021-04-06',87),('2021-04-07',37),('2021-04-08',95),('2021-04-09',60),('2021-04-10',13),('2021-04-11',50),('2021-04-12',74),('2021-04-13',46),
('2021-05-01',28),('2021-05-02',49),('2021-05-03',35),('2021-05-04',67),('2021-05-05',26),('2021-05-06',82),('2021-05-07',31),('2021-05-08',92),('2021-05-09',61),('2021-05-10',18),('2021-05-11',54),('2021-05-12',79),('2021-05-13',51),
('2021-06-01',40),('2021-06-02',59),('2021-06-03',43),('2021-06-04',66),('2021-06-05',27),('2021-06-06',85),('2021-06-07',38),('2021-06-08',94),('2021-06-09',64),('2021-06-10',17),('2021-06-11',55),('2021-06-12',77),('2021-06-13',48),
('2021-07-01',34),('2021-07-02',50),('2021-07-03',37),('2021-07-04',69),('2021-07-05',32),('2021-07-06',80),('2021-07-07',33),('2021-07-08',88),('2021-07-09',57),('2021-07-10',21),('2021-07-11',56),('2021-07-12',73),('2021-07-13',42),
('2021-08-01',41),('2021-08-02',53),('2021-08-03',39),('2021-08-04',62),('2021-08-05',23),('2021-08-06',83),('2021-08-07',29),('2021-08-08',91),('2021-08-09',59),('2021-08-10',22),('2021-08-11',51),('2021-08-12',75),('2021-08-13',44),
('2021-09-01',36),('2021-09-02',45),('2021-09-03',40),('2021-09-04',68),('2021-09-05',28),('2021-09-06',84),('2021-09-07',30),('2021-09-08',90),('2021-09-09',61),('2021-09-10',20),('2021-09-11',52),('2021-09-12',71),('2021-09-13',43),
('2021-10-01',46),('2021-10-02',58),('2021-10-03',41),('2021-10-04',63),('2021-10-05',24),('2021-10-06',82),('2021-10-07',34),('2021-10-08',86),('2021-10-09',56),('2021-10-10',14),('2021-10-11',57),('2021-10-12',70),('2021-10-13',47),
('2021-11-01',31),('2021-11-02',44),('2021-11-03',38),('2021-11-04',67),('2021-11-05',22),('2021-11-06',79),('2021-11-07',32),('2021-11-08',94),('2021-11-09',60),('2021-11-10',15),('2021-11-11',54),('2021-11-12',73),('2021-11-13',46),
('2021-12-01',29),('2021-12-02',50),('2021-12-03',42),('2021-12-04',65),('2021-12-05',25),('2021-12-06',83),('2021-12-07',30),('2021-12-08',93),('2021-12-09',58),('2021-12-10',19),('2021-12-11',52),('2021-12-12',75),('2021-12-13',48);

SELECT * FROM covid_cases ORDER BY record_date;

record_date,cases_count
2021-01-01,66
2021-01-02,41
2021-01-03,54
2021-01-04,68
2021-01-05,16
2021-01-06,90
2021-01-07,34
2021-01-08,84
2021-01-09,71
2021-01-10,14


In [0]:
%sql
with cte as (
    select year(record_date) as year, month(record_date) as month, sum(cases_count) as total_cases_per_month
    from covid_cases
    group by month(record_date), year(record_date)
)
select *,
    sum(total_cases_per_month) over(order by year, month rows between unbounded preceding and 1 preceding) as running_total,
    round(
        total_cases_per_month*100 / (sum(total_cases_per_month) over(order by year, month rows between unbounded preceding and 1 preceding)), 2
    ) as pct_change
from cte

year,month,total_cases_per_month,running_total,pct_change
2021,1,713,null,null
2021,2,677,713,94.95
2021,3,667,1390,47.99
2021,4,698,2057,33.93
2021,5,673,2755,24.43
2021,6,713,3428,20.8
2021,7,672,4141,16.23
2021,8,672,4813,13.96
2021,9,668,5485,12.18
2021,10,678,6153,11.02


#### Julia conducted a 15-day SQL learning contest. The start date of the contest was March 01, 2016, and the end date was March 15, 2016.
#### Write a SQL query to print:
- The total number of unique hackers who made at least 1 submission each day, starting on the first day of the contest.
- The hacker_id and name of the hacker who made the maximum number of submissions each day. If more than one hacker has the maximum number of submissions, print the hacker with the lowest hacker_id.
- The query should print this information for each day of the contest, sorted by date.

In [0]:
%sql
drop table if exists Submissions;
CREATE TABLE Submissions (
    submission_date DATE,
    submission_id INT PRIMARY KEY,
    hacker_id INT,
    score INT
);

INSERT INTO Submissions (submission_date, submission_id, hacker_id, score) VALUES
('2016-03-01', 8494, 20703, 0),
('2016-03-01', 22403, 53473, 15),
('2016-03-01', 23965, 79722, 60),
('2016-03-01', 30173, 36396, 70),
('2016-03-02', 34928, 20703, 0),
('2016-03-02', 38740, 15758, 60),
('2016-03-02', 42769, 79722, 25),
('2016-03-02', 44364, 79722, 60),
('2016-03-03', 45440, 20703, 0),
('2016-03-03', 49050, 36396, 70),
('2016-03-03', 50273, 79722, 5),
('2016-03-04', 50344, 20703, 0),
('2016-03-04', 51360, 44065, 90),
('2016-03-04', 54404, 53473, 65),
('2016-03-04', 61533, 79722, 15),
('2016-03-05', 72852, 20703, 0),
('2016-03-05', 74546, 38289, 0),
('2016-03-05', 76487, 62529, 0),
('2016-03-05', 82439, 36396, 10),
('2016-03-05', 90006, 36396, 40),
('2016-03-06', 90404, 20703, 0);


num_affected_rows,num_inserted_rows
21,21


In [0]:
%sql
select * from Submissions;

submission_date,submission_id,hacker_id,score
2016-03-01,8494,20703,0
2016-03-01,22403,53473,15
2016-03-01,23965,79722,60
2016-03-01,30173,36396,70
2016-03-02,34928,20703,0
2016-03-02,38740,15758,60
2016-03-02,42769,79722,25
2016-03-02,44364,79722,60
2016-03-03,45440,20703,0
2016-03-03,49050,36396,70


In [0]:
%sql
with cte as (
    select submission_date, hacker_id, count(*) as no_of_submissions,
    day(submission_date) as day_number
    from submissions
    group by submission_date, hacker_id
),
cte2 as (
    select *,
        count(*) over(partition by hacker_id order by submission_date) as till_date_submissions,
        case 
            when
                day_number = count(*) over(partition by hacker_id order by submission_date)
            then 1
            else 0
        end as unique_flag
    from cte
),
cte3 as (
    select *,
        sum(unique_flag) over(partition by submission_date) as unique_submissions,
        row_number() over(partition by submission_date order by no_of_submissions desc, hacker_id) as rank
    from cte2
)
select submission_date, unique_submissions, hacker_id
from cte3
where rank = 1
order by submission_date;

submission_date,unique_submissions,hacker_id
2016-03-01,4,20703
2016-03-02,2,79722
2016-03-03,2,20703
2016-03-04,2,20703
2016-03-05,1,36396
2016-03-06,1,20703


#### Question 15:
##### For each region find house which has won maximum no of battles. display region, house and no of wins

In [0]:
%sql
-- Create the 'king' table
drop table if exists king;
CREATE TABLE king (
    k_no INT PRIMARY KEY,
    king VARCHAR(50),
    house VARCHAR(50)
);

-- Create the 'battle' table
drop table if exists battle;
CREATE TABLE battle (
    battle_number INT PRIMARY KEY,
    name VARCHAR(100),
    attacker_king INT,
    defender_king INT,
    attacker_outcome INT,
    region VARCHAR(50),
    CONSTRAINT battle_attacker_king_fk FOREIGN KEY (attacker_king) REFERENCES king(k_no),
    CONSTRAINT battle_defender_king_fk FOREIGN KEY (defender_king) REFERENCES king(k_no)
);

INSERT INTO king (k_no, king, house) VALUES
(1, 'Robb Stark', 'House Stark'),
(2, 'Joffrey Baratheon', 'House Lannister'),
(3, 'Stannis Baratheon', 'House Baratheon'),
(4, 'Balon Greyjoy', 'House Greyjoy'),
(5, 'Mace Tyrell', 'House Tyrell'),
(6, 'Doran Martell', 'House Martell');

-- Insert data into the 'battle' table
INSERT INTO battle (battle_number, name, attacker_king, defender_king, attacker_outcome, region) VALUES
(1, 'Battle of Oxcross', 1, 2, 1, 'The North'),
(2, 'Battle of Blackwater', 3, 4, 0, 'The North'),
(3, 'Battle of the Fords', 1, 5, 1, 'The Reach'),
(4, 'Battle of the Green Fork', 2, 6, 0, 'The Reach'),
(5, 'Battle of the Ruby Ford', 1, 3, 1, 'The Riverlands'),
(6, 'Battle of the Golden Tooth', 2, 1, 0, 'The North'),
(7, 'Battle of Riverrun', 3, 4, 1, 'The Riverlands'),
(8, 'Battle of Riverrun', 1, 3, 0, 'The Riverlands');

select * from battle;

battle_number,name,attacker_king,defender_king,attacker_outcome,region
1,Battle of Oxcross,1,2,1,The North
2,Battle of Blackwater,3,4,0,The North
3,Battle of the Fords,1,5,1,The Reach
4,Battle of the Green Fork,2,6,0,The Reach
5,Battle of the Ruby Ford,1,3,1,The Riverlands
6,Battle of the Golden Tooth,2,1,0,The North
7,Battle of Riverrun,3,4,1,The Riverlands
8,Battle of Riverrun,1,3,0,The Riverlands


In [0]:
%sql
select * from king;

k_no,king,house
1,Robb Stark,House Stark
2,Joffrey Baratheon,House Lannister
3,Stannis Baratheon,House Baratheon
4,Balon Greyjoy,House Greyjoy
5,Mace Tyrell,House Tyrell
6,Doran Martell,House Martell


In [0]:
%sql
with cte as (
select * from battle b 
join king k 
on k.k_no = case when b.attacker_outcome = 1 then b.attacker_king else b.defender_king end
)
select * from (
select region, house, count(*) as wins, rank() over(partition by region order by count(*) desc) as rn
from cte 
group by region, house)
where rn = 1;

region,house,wins,rn
The North,House Stark,2,1
The Reach,House Martell,1,1
The Reach,House Stark,1,1
The Riverlands,House Baratheon,2,1


In [0]:
%sql
drop table if exists users;
CREATE TABLE users (
    USER_ID INT PRIMARY KEY,
    USER_NAME VARCHAR(20) NOT NULL,
    USER_STATUS VARCHAR(20) NOT NULL
);

drop table if exists logins;
CREATE TABLE logins (
    USER_ID INT,
    LOGIN_TIMESTAMP TIMESTAMP NOT NULL,
    SESSION_ID INT PRIMARY KEY,
    SESSION_SCORE INT,
    FOREIGN KEY (USER_ID) REFERENCES USERS(USER_ID)
);

-- Users Table
INSERT INTO USERS VALUES (1, 'Alice', 'Active');
INSERT INTO USERS VALUES (2, 'Bob', 'Inactive');
INSERT INTO USERS VALUES (3, 'Charlie', 'Active');
INSERT INTO USERS  VALUES (4, 'David', 'Active');
INSERT INTO USERS  VALUES (5, 'Eve', 'Inactive');
INSERT INTO USERS  VALUES (6, 'Frank', 'Active');
INSERT INTO USERS  VALUES (7, 'Grace', 'Inactive');
INSERT INTO USERS  VALUES (8, 'Heidi', 'Active');
INSERT INTO USERS VALUES (9, 'Ivan', 'Inactive');
INSERT INTO USERS VALUES (10, 'Judy', 'Active');

-- Logins Table 
INSERT INTO LOGINS  VALUES (1, '2026-07-15 09:30:00', 1001, 85);
INSERT INTO LOGINS VALUES (2, '2026-07-22 10:00:00', 1002, 90);
INSERT INTO LOGINS VALUES (3, '2026-08-10 11:15:00', 1003, 75);
INSERT INTO LOGINS VALUES (4, '2026-08-20 14:00:00', 1004, 88);
INSERT INTO LOGINS  VALUES (5, '2026-09-05 16:45:00', 1005, 82);
INSERT INTO LOGINS  VALUES (6, '2026-10-12 08:30:00', 1006, 77);
INSERT INTO LOGINS  VALUES (7, '2026-11-18 09:00:00', 1007, 81);
INSERT INTO LOGINS VALUES (8, '2026-12-01 10:30:00', 1008, 84);
INSERT INTO LOGINS  VALUES (9, '2026-12-15 13:15:00', 1009, 79);


-- 2024 Q1
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (1, '2026-01-10 07:45:00', 1011, 86);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (2, '2026-01-25 09:30:00', 1012, 89);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (3, '2026-02-05 11:00:00', 1013, 78);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (4, '2026-03-01 14:30:00', 1014, 91);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (5, '2026-03-15 16:00:00', 1015, 83);

INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (6, '2026-04-12 08:00:00', 1016, 80);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (7, '2026-05-18 09:15:00', 1017, 82);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (8, '2026-05-28 10:45:00', 1018, 87);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (9, '2026-06-15 13:30:00', 1019, 76);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (10, '2026-06-25 15:00:00', 1010, 92);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (10, '2026-06-26 15:45:00', 1020, 93);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (10, '2026-06-27 15:00:00', 1021, 92);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (10, '2026-06-28 15:45:00', 1022, 93);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (1, '2026-01-10 07:45:00', 1101, 86);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (3, '2026-01-25 09:30:00', 1102, 89);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (5, '2026-01-15 11:00:00', 1103, 78);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (2, '2026-11-10 07:45:00', 1201, 82);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (4, '2026-11-25 09:30:00', 1202, 84);
INSERT INTO LOGINS (USER_ID, LOGIN_TIMESTAMP, SESSION_ID, SESSION_SCORE) VALUES (6, '2026-11-15 11:00:00', 1203, 80);

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
select * from users;

USER_ID,USER_NAME,USER_STATUS
3,Charlie,Active
7,Grace,Inactive
9,Ivan,Inactive
2,Bob,Inactive
5,Eve,Inactive
1,Alice,Active
8,Heidi,Active
6,Frank,Active
4,David,Active
10,Judy,Active


In [0]:
%sql
select * from logins
order by LOGIN_TIMESTAMP;

USER_ID,LOGIN_TIMESTAMP,SESSION_ID,SESSION_SCORE
1,2026-01-10T07:45:00.000Z,1011,86
1,2026-01-10T07:45:00.000Z,1101,86
5,2026-01-15T11:00:00.000Z,1103,78
2,2026-01-25T09:30:00.000Z,1012,89
3,2026-01-25T09:30:00.000Z,1102,89
3,2026-02-05T11:00:00.000Z,1013,78
4,2026-03-01T14:30:00.000Z,1014,91
5,2026-03-15T16:00:00.000Z,1015,83
6,2026-04-12T08:00:00.000Z,1016,80
7,2026-05-18T09:15:00.000Z,1017,82


##### Question1 - 
- Find users who didn't logged in last 5 months

In [0]:
%sql
------------------------- Method 1 ------------------

-- select user_id, max(LOGIN_TIMESTAMP)
-- from logins
-- group by USER_ID
-- having max(LOGIN_TIMESTAMP) < dateadd(month, -5, getdate());


------------------------- Method 2 ------------------------------

select * from logins where USER_ID not in (
    select USER_ID from logins
    where LOGIN_TIMESTAMP > dateadd(month, -5, getdate())
)

USER_ID,LOGIN_TIMESTAMP,SESSION_ID,SESSION_SCORE


##### Question2 - 
- For business unit's quarterly analysis, calculate how many users & sessions were at each quarter. 
- Return first day of quarter, user count, session count

In [0]:
%sql
select quarter(LOGIN_TIMESTAMP) as quarter, 
    min(login_timestamp) as quarter_first_login, 
    count(SESSION_ID) as no_of_sessions, 
    count(distinct USER_ID) as no_of_users,
    date_trunc('quarter', min(LOGIN_TIMESTAMP)) as quarter_first_day
from logins
group by quarter(LOGIN_TIMESTAMP)
order by quarter(LOGIN_TIMESTAMP);

quarter,quarter_first_login,no_of_sessions,no_of_users,quarter_first_day
1,2026-01-10T07:45:00.000Z,8,5,2026-01-01T00:00:00.000Z
2,2026-04-12T08:00:00.000Z,8,5,2026-04-01T00:00:00.000Z
3,2026-07-15T09:30:00.000Z,5,5,2026-07-01T00:00:00.000Z
4,2026-10-12T08:30:00.000Z,7,6,2026-10-01T00:00:00.000Z
